# Fetch Missing Binance Data via API

Fetch 1h klines (00:00 UTC candle per day) for symbols missing from ClickHouse,
and save in the same format as the existing `binance_v2` parquet files.

In [ ]:
from datetime import UTC
from datetime import datetime
from pathlib import Path

import pandas as pd
from binance.client import Client
from binance.enums import HistoricalKlinesType

In [2]:
# --- Configuration ---
SYMBOLS = ["FTTUSDT"]  # Add more symbols here if needed
START_DATE = "2024-12-01"
END_DATE = datetime.now(UTC).strftime("%Y-%m-%d")
OUTPUT_DIR = Path("../data/binance_v2")

print(f"Symbols: {SYMBOLS}")
print(f"Date range: {START_DATE} to {END_DATE}")
print(f"Output: {OUTPUT_DIR.resolve()}")

Symbols: ['FTTUSDT']
Date range: 2024-12-01 to 2026-02-21
Output: /Users/mikey/Desktop/nautilus_trader/my_strategies/data/binance_v2


In [3]:
def fetch_1h_daily_klines(client: Client, symbol: str, start_date: str, end_date: str) -> pd.DataFrame:
    """Fetch 1h perpetual klines from Binance API, filtered to 00:00 UTC only."""
    klines = client.get_historical_klines(
        symbol=symbol,
        interval=Client.KLINE_INTERVAL_1HOUR,
        start_str=f"{start_date} 00:00:00",
        end_str=f"{end_date} 23:59:59",
        klines_type=HistoricalKlinesType.FUTURES,
    )

    if not klines:
        return pd.DataFrame()

    df = pd.DataFrame(klines, columns=[
        "timestamp", "open", "high", "low", "close", "volume",
        "close_time", "quote_volume", "trades_count", "taker_buy_volume",
        "taker_buy_quote_volume", "ignore",
    ])

    # Convert types
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True)
    df["close_time"] = pd.to_datetime(df["close_time"], unit="ms", utc=True)
    for col in ["open", "high", "low", "close", "volume", "quote_volume",
                "taker_buy_volume", "taker_buy_quote_volume"]:
        df[col] = df[col].astype(float)
    df["trades_count"] = df["trades_count"].astype(int)

    # Filter to 00:00 UTC only (1 candle per day)
    df = df[df["timestamp"].dt.hour == 0].copy()

    # Add metadata columns to match existing parquet format
    df["symbol"] = symbol
    df["exchange"] = "binance"
    df["interval"] = "1h"
    df["type"] = "PERPETUAL"

    # Reorder to match existing files
    df = df[[
        "symbol", "exchange", "interval", "timestamp", "type", "close_time",
        "open", "high", "low", "close", "volume", "quote_volume",
        "taker_buy_volume", "taker_buy_quote_volume", "trades_count",
    ]].reset_index(drop=True)

    return df

In [4]:
# Fetch data
client = Client()

for symbol in SYMBOLS:
    print(f"Fetching {symbol}...")
    df = fetch_1h_daily_klines(client, symbol, START_DATE, END_DATE)

    if df.empty:
        print(f"  No data found for {symbol}")
        continue

    print(f"  Rows: {len(df)}")
    print(f"  Date range: {df['timestamp'].min()} -> {df['timestamp'].max()}")
    display(df.head())

Fetching FTTUSDT...
  Rows: 448
  Date range: 2024-12-01 00:00:00+00:00 -> 2026-02-21 00:00:00+00:00


,symbol,exchange,interval,timestamp,type,close_time,open,high,low,close,volume,quote_volume,taker_buy_volume,taker_buy_quote_volume,trades_count
0,FTTUSDT,binance,1h,2024-12-01 00:00:00+00:00,PERPETUAL,2024-12-01 00:59:59.999000+00:00,1.59,1.59,1.59,1.59,0.0,0.0,0.0,0.0,0
1,FTTUSDT,binance,1h,2024-12-02 00:00:00+00:00,PERPETUAL,2024-12-02 00:59:59.999000+00:00,1.59,1.59,1.59,1.59,0.0,0.0,0.0,0.0,0
2,FTTUSDT,binance,1h,2024-12-03 00:00:00+00:00,PERPETUAL,2024-12-03 00:59:59.999000+00:00,1.59,1.59,1.59,1.59,0.0,0.0,0.0,0.0,0
3,FTTUSDT,binance,1h,2024-12-04 00:00:00+00:00,PERPETUAL,2024-12-04 00:59:59.999000+00:00,1.59,1.59,1.59,1.59,0.0,0.0,0.0,0.0,0
4,FTTUSDT,binance,1h,2024-12-05 00:00:00+00:00,PERPETUAL,2024-12-05 00:59:59.999000+00:00,1.59,1.59,1.59,1.59,0.0,0.0,0.0,0.0,0


In [6]:
# Compare with an existing file to make sure format matches
existing = pd.read_parquet(OUTPUT_DIR / "binance_AAVEUSDT_1h_daily_20260221_130805.parquet")
print("Existing file columns:", list(existing.columns))
print("New data columns:     ", list(df.columns))
print()
print("Existing dtypes:")
print(existing.dtypes)
print()
print("New dtypes:")
print(df.dtypes)

Existing file columns: ['symbol', 'exchange', 'interval', 'timestamp', 'type', 'close_time', 'open', 'high', 'low', 'close', 'volume', 'quote_volume', 'taker_buy_volume', 'taker_buy_quote_volume', 'trades_count']
New data columns:      ['symbol', 'exchange', 'interval', 'timestamp', 'type', 'close_time', 'open', 'high', 'low', 'close', 'volume', 'quote_volume', 'taker_buy_volume', 'taker_buy_quote_volume', 'trades_count']

Existing dtypes:
symbol                                    str
exchange                                  str
interval                                  str
timestamp                 datetime64[ns, UTC]
type                                      str
close_time                datetime64[ns, UTC]
open                                  float64
high                                  float64
low                                   float64
close                                 float64
volume                                float64
quote_volume                          float64
take

In [7]:
# Save to parquet
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for symbol in SYMBOLS:
    df = fetch_1h_daily_klines(client, symbol, START_DATE, END_DATE)
    if df.empty:
        print(f"Skipping {symbol} - no data")
        continue

    timestamp_str = datetime.now(UTC).strftime("%Y%m%d_%H%M%S")
    filename = f"binance_{symbol}_1h_daily_{timestamp_str}.parquet"
    filepath = OUTPUT_DIR / filename
    df.to_parquet(filepath, index=False)
    print(f"Saved: {filepath} ({len(df)} rows)")

Saved: ../data/binance_v2/binance_FTTUSDT_1h_daily_20260221_132337.parquet (448 rows)
